# Jigsaw puzzle — final neural solution

The task is to reconstruct a complete RGB image of size **96×96×3** from **9 scrambled patches** of size **28×28×3**.

Final approach used in this notebook:

1. A neural network predicts a **soft permutation matrix** assigning each input patch to one of the 9 grid positions.
2. Two auxiliary neural heads predict **right-neighbour** and **bottom-neighbour** relationships between patches.
3. A differentiable neural reassembly layer places the original patches into a 96×96 canvas using the predicted permutation.
4. A small convolutional refiner fills the missing borders caused by the 28×28 erosion.

This avoids a blurry pure image generator and keeps the solution aligned with the jigsaw objective.


In [1]:
# ============================================================
# Imports and reproducibility
# ============================================================
import os
import numpy as np
import tensorflow as tf
import keras
import matplotlib.pyplot as plt

from keras import layers
from keras.utils import PyDataset

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))


TensorFlow version: 2.20.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Load STL-10 unlabeled images

The images are kept as `uint8` to avoid using too much RAM. Normalization to `[0,1]` is done inside the generators, batch by batch.


In [2]:
def download_and_load_stl10():
    path = tf.keras.utils.get_file(
        'stl10_binary.tar.gz',
        origin='http://ai.stanford.edu/~acoates/stl10/stl10_binary.tar.gz',
        extract=True
    )

    base_dir = os.path.dirname(path)
    data_dir = os.path.join(base_dir, 'stl10_binary_extracted', 'stl10_binary')
    filepath = os.path.join(data_dir, 'unlabeled_X.bin')

    if not os.path.exists(filepath):
        raise FileNotFoundError(f"Could not find the binary file at {filepath}")

    print(f"Loading data from: {filepath}")

    with open(filepath, 'rb') as f:
        data = np.fromfile(f, dtype=np.uint8)
        images = np.reshape(data, (-1, 3, 96, 96))
        images = np.transpose(images, (0, 3, 2, 1))

    return images

images = download_and_load_stl10()
print("Images:", images.shape, images.dtype)


2640397119/2640397119 ━━━━━━━━━━━━━━━━━━━━ 128s 0us/step
Loading data from: /root/.keras/datasets/stl10_binary_extracted/stl10_binary/unlabeled_X.bin
Images: (100000, 96, 96, 3) uint8


In [3]:
# ============================================================
# Train / validation / test split
# ============================================================
# Keep this split clean: training must not use validation or test images.

train_images = images[:80000]
val_images = images[80000:90000]
test_images = images[90000:]

print(train_images.shape, val_images.shape, test_images.shape)


(80000, 96, 96, 3) (10000, 96, 96, 3) (10000, 96, 96, 3)


## Baseline generator and baseline MAE

This is close to the professor's baseline. It creates scrambled 28×28 patches and uses the full image as target.


In [4]:
class PatchGenerator(PyDataset):
    def __init__(self, images, batch_size=32, patch_size=32, crop_size=28, shuffle=True, **kwargs):
        super().__init__(**kwargs)
        self.images = images
        self.batch_size = batch_size
        self.patch_size = patch_size
        self.crop_size = crop_size
        self.shuffle = shuffle
        self.indices = np.arange(len(self.images))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.images) / self.batch_size))

    def __getitem__(self, idx):
        batch_indices = self.indices[idx * self.batch_size : (idx + 1) * self.batch_size]
        actual_batch_size = len(batch_indices)

        X = np.zeros((actual_batch_size, 9, self.crop_size, self.crop_size, 3), dtype="float32")
        Y = np.zeros((actual_batch_size, 96, 96, 3), dtype="float32")

        for i, img_idx in enumerate(batch_indices):
            full_img = self.images[img_idx].astype("float32") / 255.0
            Y[i] = full_img
            patches = []

            for r in range(3):
                for c in range(3):
                    y_start, x_start = r * self.patch_size, c * self.patch_size
                    patch = full_img[y_start:y_start+self.patch_size,
                                     x_start:x_start+self.patch_size, :]
                    margin = (self.patch_size - self.crop_size) // 2
                    patch = patch[margin:margin+self.crop_size,
                                  margin:margin+self.crop_size, :]
                    patches.append(patch)

            order = np.random.permutation(9)
            for slot_idx, original_pos in enumerate(order):
                X[i, slot_idx] = patches[original_pos]

        return X, Y

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)


def mean_patch_image(patches):
    B = tf.shape(patches)[0]
    mean_patch = tf.reduce_mean(patches, axis=1)
    mean_patches = tf.repeat(mean_patch[:, None, :, :, :], repeats=9, axis=1)
    out = tf.reshape(mean_patches, (B, 3, 3, 28, 28, 3))
    out = tf.transpose(out, [0, 1, 3, 2, 4, 5])
    out = tf.reshape(out, (B, 84, 84, 3))
    out = tf.image.resize(out, (96, 96))
    return out

baseline_test_generator = PatchGenerator(test_images, batch_size=64, shuffle=False)
mae_values = []
for i in range(len(baseline_test_generator)):
    x_batch, y_batch = baseline_test_generator[i]
    pred = mean_patch_image(x_batch)
    batch_mae = np.mean(np.abs(pred.numpy() - y_batch), axis=(1,2,3))
    mae_values.extend(batch_mae)

print("Baseline MAE:", np.mean(mae_values))
print("Baseline STD:", np.std(mae_values))


Baseline MAE: 0.18236528
Baseline STD: 0.05612072


# Final model: neural permutation + neighbour prediction + differentiable reconstruction

Important design choices:

- The model is not a generic image generator. It predicts a patch-to-position assignment.
- The Sinkhorn layer keeps the predicted assignment close to a valid permutation matrix.
- Right/bottom neighbour heads force the model to learn border compatibility.
- The reassembly layer places the input patches into the predicted locations inside the neural network.
- No pretrained models are used.
- The model is fully implemented in Keras and stays below 6 million trainable parameters.


In [5]:
# ============================================================
# Final data size configuration
# ============================================================
# For a final run use 40000/5000/5000 or more if Colab allows it.
# For a quick debug run, reduce TRAIN_LIMIT to 5000.

TRAIN_LIMIT = 40000
VAL_LIMIT = 5000
TEST_LIMIT = 5000
BATCH_SIZE = 32
EPOCHS = 30

train_images_final = train_images[:TRAIN_LIMIT]
val_images_final = val_images[:VAL_LIMIT]
test_images_final = test_images[:TEST_LIMIT]

print(train_images_final.shape, val_images_final.shape, test_images_final.shape)


(40000, 96, 96, 3) (5000, 96, 96, 3) (5000, 96, 96, 3)


In [6]:
class NeuralPuzzleGenerator(PyDataset):
    def __init__(
        self,
        images,
        batch_size=32,
        patch_size=32,
        crop_size=28,
        shuffle=True,
        repeat_factor=1,
        use_augmentation=False,
        **kwargs
    ):
        super().__init__(**kwargs)
        self.images = images                  # keep uint8; normalize inside __getitem__
        self.batch_size = batch_size
        self.patch_size = patch_size
        self.crop_size = crop_size
        self.shuffle = shuffle
        self.repeat_factor = repeat_factor
        self.use_augmentation = use_augmentation
        self.indices = np.arange(len(images))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.indices) * self.repeat_factor / self.batch_size))

    def augment_patch(self, patch):
        if not self.use_augmentation:
            return patch

        brightness = np.random.uniform(-0.05, 0.05)
        contrast = np.random.uniform(0.92, 1.08)
        noise = np.random.normal(0, 0.008, patch.shape)

        patch = patch + brightness
        patch = (patch - 0.5) * contrast + 0.5
        patch = patch + noise
        return np.clip(patch, 0.0, 1.0).astype(np.float32)

    def __getitem__(self, idx):
        if self.repeat_factor > 1:
            batch_idx = np.random.choice(self.indices, size=self.batch_size, replace=True)
        else:
            batch_idx = self.indices[idx*self.batch_size:(idx+1)*self.batch_size]

        bs = len(batch_idx)
        X = np.zeros((bs, 9, self.crop_size, self.crop_size, 3), dtype=np.float32)
        Y_position = np.zeros((bs, 9, 9), dtype=np.float32)
        Y_right = np.zeros((bs, 9, 9), dtype=np.float32)
        Y_bottom = np.zeros((bs, 9, 9), dtype=np.float32)
        Y_img = np.zeros((bs, 96, 96, 3), dtype=np.float32)

        for i, img_idx in enumerate(batch_idx):
            img = self.images[img_idx].astype("float32") / 255.0
            Y_img[i] = img
            patches = []

            for r in range(3):
                for c in range(3):
                    y = r * self.patch_size
                    x = c * self.patch_size
                    patch = img[y:y+self.patch_size, x:x+self.patch_size, :]
                    margin = (self.patch_size - self.crop_size) // 2
                    patch = patch[margin:margin+self.crop_size,
                                  margin:margin+self.crop_size, :]
                    patch = self.augment_patch(patch)
                    patches.append(patch)

            order = np.random.permutation(9)       # order[slot] = original position
            original_to_scrambled = np.zeros(9, dtype=np.int32)

            for slot, original_pos in enumerate(order):
                X[i, slot] = patches[original_pos]
                Y_position[i, slot, original_pos] = 1.0
                original_to_scrambled[original_pos] = slot

            # neighbour targets in scrambled-patch index space
            for pos in range(9):
                row = pos // 3
                col = pos % 3
                patch_a = original_to_scrambled[pos]

                if col < 2:
                    patch_b = original_to_scrambled[pos + 1]
                    Y_right[i, patch_a, patch_b] = 1.0

                if row < 2:
                    patch_b = original_to_scrambled[pos + 3]
                    Y_bottom[i, patch_a, patch_b] = 1.0

        return X, {
            "sinkhorn_permutation": Y_position,
            "right_neighbor": Y_right,
            "bottom_neighbor": Y_bottom,
            "reconstructed_image": Y_img,
        }

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)


train_generator_final = NeuralPuzzleGenerator(
    train_images_final,
    batch_size=BATCH_SIZE,
    repeat_factor=2,
    shuffle=True,
    use_augmentation=True,
)
val_generator_final = NeuralPuzzleGenerator(
    val_images_final,
    batch_size=BATCH_SIZE,
    repeat_factor=1,
    shuffle=False,
    use_augmentation=False,
)
test_generator_final = NeuralPuzzleGenerator(
    test_images_final,
    batch_size=BATCH_SIZE,
    repeat_factor=1,
    shuffle=False,
    use_augmentation=False,
)


In [7]:
class SinkhornLayer(layers.Layer):
    def __init__(self, iterations=25, temperature=0.35, **kwargs):
        super().__init__(**kwargs)
        self.iterations = iterations
        self.temperature = temperature

    def call(self, logits):
        logits = logits / self.temperature
        logits = logits - tf.reduce_max(logits, axis=[-1, -2], keepdims=True)
        x = tf.exp(logits)

        for _ in range(self.iterations):
            x = x / (tf.reduce_sum(x, axis=-1, keepdims=True) + 1e-8)
            x = x / (tf.reduce_sum(x, axis=-2, keepdims=True) + 1e-8)

        return x

    def get_config(self):
        config = super().get_config()
        config.update({"iterations": self.iterations, "temperature": self.temperature})
        return config


class EdgeFeatureLayer(layers.Layer):
    def __init__(self, band_size=4, **kwargs):
        super().__init__(**kwargs)
        self.band_size = band_size

    def call(self, patches):
        b = self.band_size
        top = patches[:, :, :b, :, :]
        bottom = patches[:, :, -b:, :, :]
        left = patches[:, :, :, :b, :]
        right = patches[:, :, :, -b:, :]

        top = tf.reshape(top, (-1, 9, b * 28 * 3))
        bottom = tf.reshape(bottom, (-1, 9, b * 28 * 3))
        left = tf.reshape(left, (-1, 9, 28 * b * 3))
        right = tf.reshape(right, (-1, 9, 28 * b * 3))

        return tf.concat([top, bottom, left, right], axis=-1)

    def get_config(self):
        config = super().get_config()
        config.update({"band_size": self.band_size})
        return config


class SoftReassemble96(layers.Layer):
    def __init__(self, patch_size=32, crop_size=28, **kwargs):
        super().__init__(**kwargs)
        self.patch_size = patch_size
        self.crop_size = crop_size

    def call(self, inputs):
        patches, permutation = inputs

        # permutation: (batch, input_patch_index, output_position)
        # cells:       (batch, output_position, 28, 28, 3)
        cells = tf.einsum("bkp,bkhwc->bphwc", permutation, patches)

        batch_size = tf.shape(patches)[0]
        cells_flat = tf.reshape(cells, (-1, 28, 28, 3))

        margin = (self.patch_size - self.crop_size) // 2
        cells_32 = tf.pad(cells_flat, [[0, 0], [margin, margin], [margin, margin], [0, 0]])

        grid = tf.reshape(cells_32, (batch_size, 3, 3, 32, 32, 3))
        rows = []
        for r in range(3):
            row = tf.concat([grid[:, r, 0], grid[:, r, 1], grid[:, r, 2]], axis=2)
            rows.append(row)

        return tf.concat(rows, axis=1)

    def get_config(self):
        config = super().get_config()
        config.update({"patch_size": self.patch_size, "crop_size": self.crop_size})
        return config


def transformer_block(x, num_heads=4, key_dim=48, ff_dim=384, dropout=0.1):
    attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=key_dim, dropout=dropout)(x, x)
    x = layers.Add()([x, attn])
    x = layers.LayerNormalization()(x)

    ff = layers.Dense(ff_dim, activation="relu")(x)
    ff = layers.Dropout(dropout)(ff)
    ff = layers.Dense(x.shape[-1])(ff)

    x = layers.Add()([x, ff])
    x = layers.LayerNormalization()(x)
    return x


def pairwise_neighbor_head(context, name):
    a = layers.Lambda(lambda t: tf.expand_dims(t, axis=2))(context)
    b = layers.Lambda(lambda t: tf.expand_dims(t, axis=1))(context)

    a = layers.Lambda(lambda t: tf.tile(t, [1, 1, 9, 1]))(a)
    b = layers.Lambda(lambda t: tf.tile(t, [1, 9, 1, 1]))(b)

    pair = layers.Concatenate(axis=-1)([
        a,
        b,
        layers.Subtract()([a, b]),
        layers.Multiply()([a, b])
    ])

    pair = layers.Dense(128, activation="relu")(pair)
    pair = layers.Dropout(0.10)(pair)
    pair = layers.Dense(64, activation="relu")(pair)

    scores = layers.Dense(1, activation="sigmoid")(pair)
    scores = layers.Reshape((9, 9), name=name)(scores)
    return scores


In [8]:
def build_final_neural_puzzle_model():
    inputs = keras.Input(shape=(9, 28, 28, 3), name="patches_input")

    # CNN patch encoder
    x = layers.TimeDistributed(layers.Conv2D(32, 3, padding="same", activation="relu"))(inputs)
    x = layers.TimeDistributed(layers.BatchNormalization())(x)
    x = layers.TimeDistributed(layers.MaxPooling2D())(x)

    x = layers.TimeDistributed(layers.Conv2D(64, 3, padding="same", activation="relu"))(x)
    x = layers.TimeDistributed(layers.BatchNormalization())(x)
    x = layers.TimeDistributed(layers.MaxPooling2D())(x)

    x = layers.TimeDistributed(layers.Conv2D(128, 3, padding="same", activation="relu"))(x)
    x = layers.TimeDistributed(layers.BatchNormalization())(x)

    global_features = layers.TimeDistributed(layers.GlobalAveragePooling2D())(x)

    # Explicit border features
    edge_features = EdgeFeatureLayer(band_size=4)(inputs)
    edge_features = layers.TimeDistributed(layers.Dense(128, activation="relu"))(edge_features)
    edge_features = layers.TimeDistributed(layers.Dense(64, activation="relu"))(edge_features)

    features = layers.Concatenate(axis=-1)([global_features, edge_features])
    features = layers.TimeDistributed(layers.Dense(192, activation="relu"))(features)
    features = layers.Dropout(0.15)(features)

    # Patch-to-patch reasoning
    context = transformer_block(features, num_heads=4, key_dim=48, ff_dim=384, dropout=0.1)
    context = transformer_block(context, num_heads=4, key_dim=48, ff_dim=384, dropout=0.1)

    # Position head: soft permutation matrix
    logits = layers.TimeDistributed(layers.Dense(9))(context)
    permutation = SinkhornLayer(
        iterations=25,
        temperature=0.35,
        name="sinkhorn_permutation"
    )(logits)

    # Auxiliary neighbour heads
    right_output = pairwise_neighbor_head(context, name="right_neighbor")
    bottom_output = pairwise_neighbor_head(context, name="bottom_neighbor")

    # Differentiable neural reassembly
    reassembled_canvas = SoftReassemble96(name="soft_reassembled_canvas")([inputs, permutation])

    # Small refiner only fills missing 2-pixel margins and smooths seams
    y = layers.Conv2D(32, 3, padding="same", activation="relu")(reassembled_canvas)
    y = layers.Conv2D(32, 3, padding="same", activation="relu")(y)
    y = layers.Conv2D(16, 3, padding="same", activation="relu")(y)
    reconstructed = layers.Conv2D(
        3,
        3,
        padding="same",
        activation="sigmoid",
        name="reconstructed_image"
    )(y)

    model = keras.Model(
        inputs=inputs,
        outputs={
            "sinkhorn_permutation": permutation,
            "right_neighbor": right_output,
            "bottom_neighbor": bottom_output,
            "reconstructed_image": reconstructed,
        },
        name="final_neural_jigsaw_solver"
    )

    return model


tf.keras.backend.clear_session()
model = build_final_neural_puzzle_model()
model.summary()
print("Total trainable parameters:", model.count_params())


Model: "final_neural_jigsaw_solver"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ patches_input       │ (None, 9, 28, 28, │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed    │ (None, 9, 28, 28, │        896 │ patches_input[0]… │
│ (TimeDistributed)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_1  │ (None, 9, 28, 28, │        128 │ time_distributed… │
│ (TimeDistributed)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_2  │ (None, 9, 14, 14, │          0 │ time_distributed… │
│ (TimeDistributed)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_3  │ (None, 9, 14, 14, │     18,496 │ time_distributed… │
│ (TimeDistributed)   │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_4  │ (None, 9, 14, 14, │        256 │ time_distributed… │
│ (TimeDistributed)   │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_5  │ (None, 9, 7, 7,   │          0 │ time_distributed… │
│ (TimeDistributed)   │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_6  │ (None, 9, 7, 7,   │     73,856 │ time_distributed… │
│ (TimeDistributed)   │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ edge_feature_layer  │ (None, 9, 1344)   │          0 │ patches_input[0]… │
│ (EdgeFeatureLayer)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_7  │ (None, 9, 7, 7,   │        512 │ time_distributed… │
│ (TimeDistributed)   │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_9  │ (None, 9, 128)    │    172,160 │ edge_feature_lay… │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_8  │ (None, 9, 128)    │          0 │ time_distributed… │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_10 │ (None, 9, 64)     │      8,256 │ time_distributed… │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 9, 192)    │          0 │ time_distributed… │
│ (Concatenate)       │                   │            │ time_distributed… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_11 │ (None, 9, 192)    │     37,056 │ concatenate[0][0] │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 9, 192)    │          0 │ time_distributed… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 9, 192)    │    148,224 │ dropout[0][0],    │
│ (MultiHeadAttentio… │                   │            │ dropout[0][0]   

 Total params: 1,136,110 (4.33 MB)

 Trainable params: 1,135,662 (4.33 MB)

 Non-trainable params: 448 (1.75 KB)

Total trainable parameters: 1136110


In [9]:
# The assignment requires fewer than 6 million trainable parameters.
assert model.count_params() < 6_000_000, "Model exceeds the 6M parameter limit"


In [10]:
model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=3e-4, weight_decay=1e-4),
    loss={
        "sinkhorn_permutation": "categorical_crossentropy",
        "right_neighbor": "binary_crossentropy",
        "bottom_neighbor": "binary_crossentropy",
        "reconstructed_image": "mae",
    },
    loss_weights={
        # High weight on the permutation because the actual task is jigsaw ordering.
        "sinkhorn_permutation": 6.0,
        "right_neighbor": 1.0,
        "bottom_neighbor": 1.0,
        # Image loss is still useful for final MAE and border completion.
        "reconstructed_image": 0.7,
    },
    metrics={
        "sinkhorn_permutation": ["accuracy"],
        "right_neighbor": ["binary_accuracy"],
        "bottom_neighbor": ["binary_accuracy"],
        "reconstructed_image": [keras.metrics.MeanAbsoluteError(name="mae")],
    }
)

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_sinkhorn_permutation_accuracy",
        patience=8,
        restore_best_weights=True,
        mode="max"
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_sinkhorn_permutation_accuracy",
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        mode="max",
        verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        "best_final_neural_jigsaw_solver.weights.h5",
        monitor="val_sinkhorn_permutation_accuracy",
        save_best_only=True,
        save_weights_only=True,
        mode="max"
    )
]


In [ ]:
history = model.fit(
    train_generator_final,
    validation_data=val_generator_final,
    epochs=EPOCHS,
    callbacks=callbacks
)


Epoch 1/30


In [ ]:
# Training curves
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["sinkhorn_permutation_accuracy"], label="train permutation acc")
plt.plot(history.history["val_sinkhorn_permutation_accuracy"], label="val permutation acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.title("Patch position accuracy")

plt.subplot(1, 2, 2)
plt.plot(history.history["reconstructed_image_mae"], label="train image MAE")
plt.plot(history.history["val_reconstructed_image_mae"], label="val image MAE")
plt.xlabel("Epoch")
plt.ylabel("MAE")
plt.legend()
plt.title("Image reconstruction MAE")

plt.tight_layout()
plt.show()


## Final evaluation

The official metric is MAE over the reconstructed 96×96 image. I also report the patch-position accuracy to show whether the model is actually learning the puzzle ordering.


In [ ]:
def evaluate_final_model(model, generator):
    image_maes = []
    patch_accs = []
    full_puzzle_accs = []

    for i in range(len(generator)):
        x_batch, y_batch = generator[i]
        preds = model.predict(x_batch, verbose=0)

        pred_img = preds["reconstructed_image"]
        true_img = y_batch["reconstructed_image"]

        batch_mae = np.mean(np.abs(pred_img - true_img), axis=(1, 2, 3))
        image_maes.extend(batch_mae)

        pred_pos = np.argmax(preds["sinkhorn_permutation"], axis=-1)
        true_pos = np.argmax(y_batch["sinkhorn_permutation"], axis=-1)

        patch_accs.extend(np.mean(pred_pos == true_pos, axis=1))
        full_puzzle_accs.extend(np.all(pred_pos == true_pos, axis=1))

    return {
        "test_mae": float(np.mean(image_maes)),
        "test_std": float(np.std(image_maes)),
        "patch_position_accuracy": float(np.mean(patch_accs)),
        "full_puzzle_accuracy": float(np.mean(full_puzzle_accs)),
    }

final_results = evaluate_final_model(model, test_generator_final)
print(final_results)


In [ ]:
# Standard Keras evaluation as additional check
keras_results = model.evaluate(test_generator_final, verbose=1)
print("Keras evaluation:")
print(keras_results)
print(model.metrics_names)


## Visual inspection

The first column shows the scrambled input, the second column shows the model prediction, and the third column shows the true image.


In [ ]:
def plot_scrambled_input(x):
    return x.reshape(3, 3, 28, 28, 3).transpose(0, 2, 1, 3, 4).reshape(84, 84, 3)

x_batch, y_batch = test_generator_final[0]
preds = model.predict(x_batch, verbose=0)

n = 5
plt.figure(figsize=(12, 4*n))

for i in range(n):
    shuffled = plot_scrambled_input(x_batch[i])
    prediction = preds["reconstructed_image"][i]
    ground_truth = y_batch["reconstructed_image"][i]

    pred_pos = np.argmax(preds["sinkhorn_permutation"][i], axis=-1)
    true_pos = np.argmax(y_batch["sinkhorn_permutation"][i], axis=-1)
    patch_acc = np.mean(pred_pos == true_pos)

    plt.subplot(n, 3, 3*i + 1)
    plt.imshow(shuffled)
    plt.title("Scrambled patches")
    plt.axis("off")

    plt.subplot(n, 3, 3*i + 2)
    plt.imshow(prediction)
    plt.title(f"Model reconstruction\nPatch acc={patch_acc:.2f}")
    plt.axis("off")

    plt.subplot(n, 3, 3*i + 3)
    plt.imshow(ground_truth)
    plt.title("Ground truth")
    plt.axis("off")

plt.tight_layout()
plt.show()


## Save weights

Upload the produced `.weights.h5` file to Google Drive, share it, and place the resulting `gdown` command in the cell below.


In [ ]:
model.save_weights("final_neural_jigsaw_solver.weights.h5")
print("Weights saved as final_neural_jigsaw_solver.weights.h5")


In [ ]:
# Example format after uploading the weights to Google Drive:
# !gdown --id YOUR_FILE_ID -O final_neural_jigsaw_solver.weights.h5

# To verify loading works:
# reloaded_model = build_final_neural_puzzle_model()
# reloaded_model.load_weights("final_neural_jigsaw_solver.weights.h5")
# print("Weights loaded correctly")


## Final comments

The final model treats the task as a jigsaw puzzle ordering problem. It predicts a soft permutation matrix assigning each input patch to a spatial location, uses auxiliary neighbour heads to learn patch-to-patch compatibility, and reconstructs the image through a differentiable neural reassembly layer followed by a small convolutional border-completion module.

The model satisfies the assignment constraints: it is fully neural, uses no pretrained model, remains below 6 million trainable parameters, is implemented in Keras, and reports both MAE and standard deviation on the test set.
